# 03 – Datenvalidierung

Dieses Notebook validiert den bereinigten Datensatz in zwei Runden:
1. Nach der Imputation (Rohdaten-Check)
2. Nach der Typkonvertierung (Schema-Check)

Validierungsregeln sind in `config.py` als Pandera-Schema definiert.

In [1]:
import sys
import polars as pl
from pathlib import Path

current_dir = Path('..') / '03_src'
sys.path.append(str(current_dir))
root_dir = Path('..')

from config import Schema
from data_validation import double_numbers, detect_outliers, check_data_quality
from data_cleaner import check_amount_nulls

## 3.1 Daten laden

In [2]:
data_combined_cleaned = pl.read_parquet(
    str(root_dir / '02_data' / 'processed' / 'combined_data_cleaned.parquet')
)
print(f'Zeilen: {len(data_combined_cleaned)} | Spalten: {len(data_combined_cleaned.columns)}')

Zeilen: 937456 | Spalten: 140


## 3.2 Duplikaten-Check

In [3]:
double_numbers(data_combined_cleaned)

Anzahl der Zeilen die Duplikate sind: 0


0

## 3.3 Ausreißer-Erkennung

In [4]:
detect_outliers(data_combined_cleaned)


--- Detaillierte Ausreißer-Analyse (IQR) ---
kwh_returned_total                            |   230402 ( 24.58%)
   -> Extremwerte (Top 10 absteigend): [223.36, 221.97, 215.98, 215.18, 214.59, 213.65, 212.76, 212.15, 209.0, 208.2]

coolingdegree_us_daily                        |   186242 ( 19.87%)
   -> Extremwerte (Top 10 absteigend): [10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6]

precipitation_total_daily                     |   122357 ( 13.05%)
   -> Extremwerte (Top 10 absteigend): [79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6]

building_floorareaheated_total                |    90221 (  9.62%)
   -> Extremwerte (Top 10 absteigend): [455.0, 455.0, 455.0, 455.0, 455.0, 455.0, 455.0, 455.0, 455.0, 455.0]

building_constructionyear                     |    84145 (  8.98%)
   -> Extremwerte (Top 10 absteigend): [2020, 2020, 2020, 2020, 2020, 2020, 2020, 2020, 2020, 2020]

swissix_base                                  |    70402 (  7.51%)
   -> Extremwerte (

[{'spalte': 'kwh_returned_total',
  'anzahl': 230402,
  'prozent': 24.57736683108327,
  'extremwerte': [223.36,
   221.97,
   215.98,
   215.18,
   214.59,
   213.65,
   212.76,
   212.15,
   209.0,
   208.2]},
 {'spalte': 'coolingdegree_us_daily',
  'anzahl': 186242,
  'prozent': 19.866745745933674,
  'extremwerte': [10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6, 10.6]},
 {'spalte': 'precipitation_total_daily',
  'anzahl': 122357,
  'prozent': 13.052025908415969,
  'extremwerte': [79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6, 79.6]},
 {'spalte': 'building_floorareaheated_total',
  'anzahl': 90221,
  'prozent': 9.624025020907649,
  'extremwerte': [455.0,
   455.0,
   455.0,
   455.0,
   455.0,
   455.0,
   455.0,
   455.0,
   455.0,
   455.0]},
 {'spalte': 'building_constructionyear',
  'anzahl': 84145,
  'prozent': 8.975887935007083,
  'extremwerte': [2020, 2020, 2020, 2020, 2020, 2020, 2020, 2020, 2020, 2020]},
 {'spalte': 'swissix_base',
  'anzahl': 70402,
  'prozent'

## 3.4 Schema-Validierung (Pandera)

In [5]:
check_data_quality(data_combined_cleaned, Schema)

✅ Validierung erfolgreich: Der Dataframe entspricht dem Schema!


timestamp,timestamp_local,date,year_str,household_id,group_assignment,affects_timepoint,kwh_received_total,kwh_received_heatpump,kwh_received_other,kwh_returned_total,group,weather_id,installation_haspvsystem,protocols_available,protocols_hasmultiplevisits,protocols_reportids,metadata_available,smartmeterdata_available_15min,smartmeterdata_available_daily,smartmeterdata_available_monthly,temperature_avg_daily,temperature_max_daily,temperature_min_daily,heatingdegree_sia_daily,heatingdegree_us_daily,coolingdegree_us_daily,humidity_avg_daily,precipitation_total_daily,sunshine_duration_daily,timestamp_local_right,report_id,household_id_right,visit_year,visit_date,building_type,building_housingunits,…,heatpump_groundsource_currentpressure,heatpump_groundsource_currentpressure_okay,heatpump_groundsource_currenttemperature,heatpump_groundsource_currenttemperature_okay,heatpump_heatingcurvesetting_toohigh_beforevisit,heatpump_heatingcurvesetting_changed,heatpump_heatingcurvesetting_outside20_beforevisit,heatpump_heatingcurvesetting_outside0_beforevisit,heatpump_heatingcurvesetting_outsideminus8_beforevisit,heatpump_heatingcurvesetting_outside20_aftervisit,heatpump_heatingcurvesetting_outside0_aftervisit,heatpump_heatingcurvesetting_outsideminus8_aftervisit,heatpump_heatinglimitsetting_toohigh_beforevisit,heatpump_heatinglimitsetting_changed,heatpump_heatinglimitsetting_beforevisit,heatpump_heatinglimitsetting_aftervisit,heatpump_nightsetbacksetting_activated_beforevisit,heatpump_nightsetbacksetting_activated_aftervisit,dhw_temperaturesetting_categorization,dhw_temperaturesetting_changed,dhw_temperaturesetting_beforevisit,dhw_temperaturesetting_aftervisit,dhw_storage_lastdescaling_toolongago,dhw_storage_lastdescaling_year,heatdistribution_expansiontank_pressure_categorization,heatdistribution_expansiontank_pressure_actual,heatdistribution_expansiontank_pressure_target,heatdistribution_expansiontank_systemheight,heatdistribution_circulation_pumpstageposition_changed,heatdistribution_circulation_pumpstageposition_beforevisit,heatdistribution_circulation_pumpstageposition_aftervisit,heatdistribution_recommendation_insulatepipes,heatdistribution_recommendation_installthermostaticvalve,heatdistribution_recommendation_installrpmvalve,visit_date_date,year_str_right,swissix_base
"datetime[μs, UTC]","datetime[μs, Europe/Zurich]",date,str,str,str,str,f64,f64,f64,f64,str,str,bool,bool,bool,str,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,"datetime[μs, Europe/Zurich]",i64,i64,i64,date,str,f64,…,f64,bool,f64,bool,bool,bool,f64,f64,f64,f64,f64,f64,bool,bool,i64,i64,bool,bool,str,bool,f64,i64,bool,i64,str,f64,f64,f64,bool,i64,i64,bool,bool,bool,date,str,f64
2019-03-02 23:59:59 UTC,2019-03-03 00:59:59 CET,2019-03-03,"""2019""","""100101""","""control""","""unknown""",18.33,0.0,null,8.64,"""control""","""z6I""",true,false,false,null,true,true,true,true,8.4,13.8,2.9,11.6,9.9,0.0,65.9,0.0,6.3,2019-03-03 00:00:00 CET,null,null,null,null,"""Unbekannt""",null,…,null,false,null,false,false,false,null,null,null,null,null,null,false,false,null,null,false,false,"""Unbekannt""",false,null,null,false,null,"""Unbekannt""",null,null,null,false,null,null,false,false,false,null,null,22.79
2019-03-03 23:59:59 UTC,2019-03-04 00:59:59 CET,2019-03-04,"""2019""","""100101""","""control""","""unknown""",15.03,0.0,null,9.04,"""control""","""z6I""",true,false,false,null,true,true,true,true,7.8,13.8,5.0,12.2,10.5,0.0,54.2,2.6,1.3,2019-03-04 00:00:00 CET,null,null,null,null,"""Unbekannt""",null,…,null,false,null,false,false,false,null,null,null,null,null,null,false,false,null,null,false,false,"""Unbekannt""",false,null,null,false,null,"""Unbekannt""",null,null,null,false,null,null,false,false,false,null,null,37.938333
2019-03-04 23:59:59 UTC,2019-03-05 00:59:59 CET,2019-03-05,"""2019""","""100101""","""control""","""unknown""",16.69,0.0,null,4.57,"""control""","""z6I""",true,false,false,null,true,true,true,true,7.4,11.9,2.7,12.6,10.9,0.0,55.5,0.2,7.7,2019-03-05 00:

## 3.5 Nullwerte im finalen Datensatz

In [6]:
check_amount_nulls(data_combined_cleaned)


---------------------------------------------
TOP 20 SPALTEN MIT MEISTEN NULLS
Datensatz: 937,456 Zeilen | 140 Spalten
Spalten mit Nulls: 20
---------------------------------------------
shape: (20, 3)
┌─────────────────────────────────┬────────────┬───────┐
│ spalte                          ┆ null_count ┆ pct   │
│ ---                             ┆ ---        ┆ ---   │
│ str                             ┆ u32        ┆ f64   │
╞═════════════════════════════════╪════════════╪═══════╡
│ visit_date_date                 ┆ 937456     ┆ 100.0 │
│ year_str_right                  ┆ 937456     ┆ 100.0 │
│ heatpump_groundsource_brinecir… ┆ 935674     ┆ 99.81 │
│ heatdistribution_circulation_p… ┆ 934015     ┆ 99.63 │
│ heatdistribution_circulation_p… ┆ 933978     ┆ 99.63 │
│ heatpump_groundsource_brinecir… ┆ 933780     ┆ 99.61 │
│ heatpump_groundsource_brinecir… ┆ 933399     ┆ 99.57 │
│ heatpump_groundsource_currentp… ┆ 933165     ┆ 99.54 │
│ heatpump_groundsource_currentt… ┆ 928403     ┆ 99.03 │

spalte,null_count,pct
str,u32,f64
"""visit_date_date""",937456,100.0
"""year_str_right""",937456,100.0
"""heatpump_groundsource_brinecir…",935674,99.81
"""heatdistribution_circulation_p…",934015,99.63
"""heatdistribution_circulation_p…",933978,99.63
…,…,…
"""heatpump_heatingcurvesetting_o…",906130,96.66
"""heatpump_heatinglimitsetting_b…",905064,96.54
"""heatpump_installation_controll…",902899,96.31
